# SLR Selection Inference 

This notebook demonstrates how to load a dataset from the **SYNERGY** collection and run evaluations using our local FastAPI endpoints (`/slr-selection-generate-per-criteria`, and `/postprocessing`).

## 1. Load Dataset from SYNERGY

The easiest way to get the SYNERGY dataset is via the `synergy-dataset` Python package. We will install the package and use its CLI command `python -m synergy_dataset get` to download and build the datasets.

In [1]:
# Install the package
!pip install synergy-dataset

# Download and build the SYNERGY dataset
!python -m synergy_dataset get

Defaulting to user installation because normal site-packages is not writeable
^C


## 2. Preprocessing

### 2.1 Dataset Selection

Specify the names of the datasets from the SYNERGY repository you wish to use as a list of strings. The code below will iterate through this list to load each corresponding CSV file into a pandas DataFrame.

In [ ]:
import pandas as pd
from synergy_dataset import Dataset

# List of dataset names to load
selected_datasets = [
    "Appelbaum_2016",
    # "Bannach-Brown_2014",
    # "Cohen_2006_ACEInhibitors"
]

datasets = {}

for ds_name in selected_datasets:
    try:
        # The synergy_dataset package provides a convenient Dataset object
        ds = Dataset(ds_name)
        datasets[ds_name] = ds.to_frame()
        print(f"Successfully loaded '{ds_name}' with {len(datasets[ds_name])} rows.")
    except Exception as e:
        print(f"Dataset '{ds_name}' could not be loaded. Error: {e}")

# Display a preview of the first loaded dataset
if len(datasets) > 0:
    first_ds_name = list(datasets.keys())[0]
    print(f"\nData preview for {first_ds_name}:")
    display(datasets[first_ds_name][['title', 'abstract', 'label_included']].head())

#### Alternative: Load Dataset from Local Directory

If you already have the datasets stored locally (for example, in a `synergy-dataset-local` folder), you can load them directly using `pandas` without relying on the `synergy_dataset` package.

In [2]:
import os
import json
import pandas as pd

# Define the local directory path
local_dataset_dir = "synergy-dataset-local"
criteria_file = os.path.join(local_dataset_dir, "criterion", "synergy_criteria.jsonl")

# Load available author_names from synergy_criteria.jsonl
available_datasets = []
criteria_dict = {}

if os.path.exists(criteria_file):
    with open(criteria_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line.strip())
            author = data.get("author_name")
            if author:
                available_datasets.append(author)
                criteria_dict[author] = data.get("list_criteria", [])
    print(f"Found {len(available_datasets)} datasets in criteria file.")
    print("Available options:", available_datasets)
else:
    print(f"Criteria file not found at {criteria_file}!")

# Select dataset(s) based on author_name from the criteria file
# Change the elements in this list to the author_names available in the output above
selected_local_datasets = [
    "Appenzeller-Herzog_2019" # Example selected dataset
]

local_datasets = {}

for ds_name in selected_local_datasets:
    # Check path: <dataset_dir>/<name>/<name>.csv or <dataset_dir>/<name>.csv
    csv_path = os.path.join(local_dataset_dir, ds_name, f"{ds_name}.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(local_dataset_dir, f"{ds_name}.csv")
        
    if os.path.exists(csv_path):
        local_datasets[ds_name] = pd.read_csv(csv_path)
        print(f"\nSuccessfully loaded '{ds_name}' from local directory with {len(local_datasets[ds_name])} rows.")
        
        # Print list of criteria for this author
        if ds_name in criteria_dict:
            print(f"Criteria for '{ds_name}':")
            for c in criteria_dict[ds_name]:
                print(f" - {c}")
    else:
        print(f"\nDataset '{ds_name}' not found at {csv_path}.")

# Display a preview of the first loaded local dataset
if len(local_datasets) > 0:
    first_local_ds = list(local_datasets.keys())[0]
    print(f"\nData preview for {first_local_ds}:")
    display(local_datasets[first_local_ds][['title', 'abstract', 'label_included']].head())

Found 10 datasets in criteria file.
Available options: ['Appenzeller-Herzog_2019', 'Bos_2018', 'Donners_2021', 'Jeyaraman_2020', 'Leenaars_2020', 'Meijboom_2021', 'Muthu_2021', 'Oud_2018', 'van_de_Schoot_2018', 'Wolters_2018']

Successfully loaded 'Appenzeller-Herzog_2019' from local directory with 2873 rows.
Criteria for 'Appenzeller-Herzog_2019':
 - Patients with Wilson's Disease of any age or stage 
 - Study drug has to be one of four established therapies, namely DPen, trientine, TTM or Zn.
 - Control could be placebo, no treatment or any other treatment that does not include the respective study drug
 - Concomitant therapies had to be identical in the compared treatment arms
 - Combination therapy regimens that include the respective monotherapy drug are not considered
 - Prospective or retrospective studies reported
 - Randomized, non-randomized controlled trials and comparative observational studies
 - Animal studies, case reports, case series, cross‐sectional studies, before‐af

,title,abstract,label_included
0,Clinical aspects of Wilson's disease.,NaN,0
1,[Hepatic changes in the neurological form of W...,NaN,0
2,Copper in medicine,Copper has been found to be causative in sever...,0
3,HEPATOLENTICULAR DEGENERATION (WILSON'S DISEAS...,NaN,0
4,[Lupus erythematosus due to penicillamine asso...,NaN,0


### 2.2 Data Cleaning

In this stage, we perform data cleaning before using it for inference. The cleaning addresses the following conditions:
1. **Incomplete Title & Abstract**: Dropped if there are missing values (`NaN` or empty text).
2. **Missing Label**: Dropped if `label_included` has missing values.
3. **Data Duplication**: Dropped if there are redundant entries with identical titles and abstracts.

In [3]:
import pandas as pd

# Choose dataset source (Prioritize local_datasets from the previous step if available)
data_source = local_datasets if 'local_datasets' in locals() and len(local_datasets) > 0 else datasets

cleaned_datasets = {}

for ds_name, df in data_source.items():
    original_len = len(df)
    
    # 1. Drop data if 'title' or 'abstract' is empty / incomplete (missing)
    # Using how='any' drops the row if either title or abstract is missing. 
    # (If you want to drop only if "both" are missing, change to how='all')
    df_clean = df.dropna(subset=['title', 'abstract'], how='any')
    
    # 2. Drop data if the label ('label_included') is missing
    if 'label_included' in df_clean.columns:
        df_clean = df_clean.dropna(subset=['label_included'])
        
    # 3. Drop duplicate data (duplicate & redundant)
    # Duplicates are identified if both 'title' and 'abstract' have identical values.
    df_clean = df_clean.drop_duplicates(subset=['title', 'abstract'])
    
    # Save the cleaned dataset back to the dictionary
    cleaned_datasets[ds_name] = df_clean
    
    new_len = len(df_clean)
    print(f"Dataset '{ds_name}':")
    print(f" - Original rows: {original_len}")
    print(f" - Cleaned rows : {new_len}")
    print(f" - Dropped rows : {original_len - new_len}\n")

# Display a preview of one of the cleaned datasets
if len(cleaned_datasets) > 0:
    first_clean_ds = list(cleaned_datasets.keys())[0]
    print(f"Preview data '{first_clean_ds}' (Cleaned):")
    display(cleaned_datasets[first_clean_ds][['title', 'abstract', 'label_included']].head())

Dataset 'Appenzeller-Herzog_2019':
 - Original rows: 2873
 - Cleaned rows : 2191
 - Dropped rows : 682

Preview data 'Appenzeller-Herzog_2019' (Cleaned):


,title,abstract,label_included
2,Copper in medicine,Copper has been found to be causative in sever...,0
6,Wilson's disease in pregnancy,Patients with Wilson's disease contemplating p...,0
7,Roles of metallothionein in copper homeostasis...,Metallothionein (MT) protects the body from bo...,0
9,Refractory rickets due to Fanconi′s Syndrome s...,Renal tubular disorders are an important cause...,0
10,Effects of Anticopper Therapy on Hepatocellula...,Liver biopsy specimens from 7 patients with Wi...,0


## 3. Classification Execution

In this section, we execute the inference by hitting the local FastAPI endpoint `/slr-selection-generate-per-criteria`. 

**Note:** Please ensure your local API server is running before executing this cell. You can start the server by running the following command in your terminal:
```bash
uvicorn main:app --reload
```

In [ ]:
import requests
import json
import os

# Configuration for the API
URL = "http://127.0.1:8000"# Base URL for the API (localhost)-> Change this if your API is hosted elsewhere (e.g., on a cloud server or different port)
ENDPOINT = f"{URL}/slr-selection-generate-per-criteria" # Endpoint for the API
MODEL_NAME = "garage-bAInd/Platypus2-7B" # Options: "garage-bAInd/Platypus2-7B", "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN") # Import your HuggingFace token from environment variable (.env)
TEMPERATURE = 0.0
MAX_LENGTH = 150
ENABLE_GPU = True
DEBUG = True

inference_results = {}

for ds_name, df in cleaned_datasets.items():
    print(f"--- Executing inference for dataset: {ds_name} ---")
    
    # Retrieve criteria for the current dataset (collected in the preprocessing step)
    criteria_list = criteria_dict.get(ds_name, [])
    if not criteria_list:
        print(f"Warning: No criteria found for {ds_name}. Skipping...")
        continue
        
    ds_results = []
    
    # NOTE: We are processing only the first 3 rows here for demonstration.
    # To process the entire dataset, remove `.head(3)` and use simply `df.iterrows()`
    for idx, row in df.head(3).iterrows():
        title = row['title']
        abstract = row['abstract']
        
        payload = {
            "title": title,
            "abstract": abstract,
            "criteria": criteria_list,
            "model_name": MODEL_NAME,
            "hf_token": HF_TOKEN,
            "temperature": TEMPERATURE,
            "max_length": MAX_LENGTH,
            "enable_gpu": ENABLE_GPU,
            "debug": DEBUG
        }
        
        try:
            response = requests.post(ENDPOINT, json=payload)
            response.raise_for_status() # Raise an exception for bad status codes
            res_data = response.json()
            
            # Store the result securely
            ds_results.append({
                "original_id": idx,
                "title": title,
                "api_response": res_data
            })
            
            print(f"[{idx}] Success: Inference completed for this paper.")
            
        except requests.exceptions.RequestException as e:
            print(f"[{idx}] Error during API request: {e}")
            
    inference_results[ds_name] = ds_results
    print(f"Completed {len(ds_results)} inferences for {ds_name}.\n")

# Display a pretty-printed sample of the first response
if len(inference_results) > 0:
    first_res_ds = list(inference_results.keys())[0]
    if len(inference_results[first_res_ds]) > 0:
        sample_response = inference_results[first_res_ds][0]['api_response']
        print(f"Sample API Response output from '{first_res_ds}':")
        print(json.dumps(sample_response, indent=2))

## 4. Postprocessing

After obtaining the Likert scores from the inference stage, we use the `postprocessing` endpoint to filter and map those numeric scores into final binary labels (`relevant` or `irrelevant`). 

By defining a `threshold` and an aggregation `logic` (`AND` / `OR`), the API will evaluate the scores from each criterion to form the final publication label.

In [ ]:
# Configuration for Postprocessing
POSTPROCESSING_ENDPOINT = f"{URL}/postprocessing"
THRESHOLD = 3.0  # Papers scoring >= 3.0 are considered relevant
LOGIC = "AND"    # Logic aggregation if multiple criteria exist ("AND" or "OR")

postprocessing_results = {}

for ds_name, ds_results in inference_results.items():
    print(f"--- Executing postprocessing for dataset: {ds_name} ---")
    
    ps_results = []
    
    for row in ds_results:
        idx = row['original_id']
        title = row['title']
        scores = row['api_response'].get('likert-score', [])
        
        payload = {
            "likert-score": scores,
            "threshold": THRESHOLD,
            "logic": LOGIC
        }
        
        try:
            response = requests.post(POSTPROCESSING_ENDPOINT, json=payload)
            response.raise_for_status()
            res_data = response.json()
            
            ps_results.append({
                "original_id": idx,
                "title": title,
                "scores": scores,
                "postprocessing_response": res_data
            })
            
            print(f"[{idx}] Processed - Final Label: {res_data.get('label')}")
            
        except requests.exceptions.RequestException as e:
            print(f"[{idx}] Error during API request: {e}")
            
    postprocessing_results[ds_name] = ps_results
    print(f"Completed {len(ps_results)} postprocessing evaluations for {ds_name}.\n")

# Display a pretty-printed sample of the first response
if len(postprocessing_results) > 0:
    first_res_ds = list(postprocessing_results.keys())[0]
    if len(postprocessing_results[first_res_ds]) > 0:
        sample_response = postprocessing_results[first_res_ds][0]
        print(f"Sample Postprocessing output from '{first_res_ds}':")
        print(json.dumps(sample_response, indent=2))

## 5. Export Results

Finally, we export the postprocessing results (including scores and final labels) into CSV files for further analysis or reporting. The files will be saved in the `export_results` directory.

In [ ]:
import os
import pandas as pd

export_dir = "export_results"
os.makedirs(export_dir, exist_ok=True)

for ds_name, ps_results in postprocessing_results.items():
    if not ps_results:
        print(f"No results to export for '{ds_name}'.")
        continue

    # Convert the results list of dictionaries to a structured format for CSV
    records = []
    for res in ps_results:
        row = {
            "original_id": res["original_id"],
            "title": res["title"],
            "scores": res["scores"],
            "label_per_criteria": res["postprocessing_response"].get("label_per_criteria"),
            "final_label": res["postprocessing_response"].get("label")
        }
        records.append(row)
        
    df_export = pd.DataFrame(records)
    export_path = os.path.join(export_dir, f"{ds_name}_results.csv")
    
    # Export to CSV
    df_export.to_csv(export_path, index=False)
    print(f"Exported {len(records)} results for '{ds_name}' to {export_path}")

# Display a preview of the first exported file
if len(postprocessing_results) > 0:
    first_ds = list(postprocessing_results.keys())[0]
    first_csv_path = os.path.join(export_dir, f"{first_ds}_results.csv")
    if os.path.exists(first_csv_path):
        print(f"\Preview of {first_csv_path}:")
        display(pd.read_csv(first_csv_path).head())